# Building with the Claude API

## Structured Output

### Environment setup

In [1]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"

### Helper functions

In [8]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)

    return message.content[0].text

### Controlling output

In [9]:
messages = []

add_user_message(messages, "Count from 1 to 10")

chat(messages, stop_sequences=["5", "3, 4"])

'1, 2, '

> **Note**: The following technique can be used with any structured data, not just JSON.

In [11]:
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")

# the assistant message is used to guide Claude's response
# think of text completion continuing at the provided string
# here, the LLM resumes text generation after the '```json' string
# the LLM (assistant) will then naturally want to close the '```json' block with '```'
add_assistant_message(messages, "```json")  # start with this

# we can use the expected '```' string as a stop sequence
text = chat(messages, stop_sequences=["```"])  # end with this
text

'\n{\n  "Name": "OrderProcessingRule",\n  "EventPattern": {\n    "source": ["myapp.orders"],\n    "detail-type": ["Order Placed"]\n  },\n  "Targets": [\n    {\n      "Id": "1",\n      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:ProcessOrder"\n    }\n  ]\n}\n'

In [13]:
import json

json.loads(text.strip())

{'Name': 'OrderProcessingRule',
 'EventPattern': {'source': ['myapp.orders'], 'detail-type': ['Order Placed']},
 'Targets': [{'Id': '1',
   'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:ProcessOrder'}]}

In [7]:
messages = []

add_user_message(messages, "Write a 1 sentence description of a fake database")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end="")

FakeDataGen is a synthetic database containing 50,000 realistic but entirely fabricated customer records with names, addresses, purchase histories, and demographic information designed for software testing and development purposes.